#### Cumulative Tracking Lines [in use]

针对1小时级别视频的处理策略：将推理和渲染分离。

**Stage 1**：YOLO tracking → 保存轨迹坐标到 CSV（跳帧加速，只跑一次）  
**Stage 2**：读取 CSV → 渲染累积轨迹图（随时调整样式，无需重跑模型）

**主要优化手段：**
| 手段 | 效果 |
|---|---|
| `FRAME_STRIDE = 3` | 每3帧处理一次，速度提升3× |
| `imgsz = 640` | 推理分辨率降低，速度明显提升 |
| 推理/渲染分离 | 崩溃可续，样式调整不重跑 |
| `verbose=False` | 关闭输出日志，略微提速 |

In [ ]:
import cv2
import csv
import os
import time

import numpy as np
import pandas as pd
import torch

import PIL.Image as PILImage
from IPython.display import display as ipython_display
from matplotlib import colors as mcolors
from ultralytics import YOLO

In [15]:
# 自动选择 GPU / CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HALF   = DEVICE == "cuda"   # GPU 上启用 FP16（速度再提升约 2×）
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
print(f"FP16  : {HALF}")

Device: cuda (NVIDIA GeForce RTX 3060 Laptop GPU)
FP16  : True


In [ ]:
def video_info(video_path):
    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f"无法打开: {video_path}"

    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    sec    = frames / fps if fps > 0 else 0
    cap.release()

    print(f"分辨率 : {w}×{h}")
    print(f"帧率   : {fps:.2f} fps")
    print(f"总帧数 : {frames:,}")
    print(f"时长   : {int(sec)//60}m {int(sec)%60:02d}s  ({sec:.1f}s)")

video_info("data/video/sample_video/视角一Fri12：00-12：15-sample.mp4")

In [ ]:
"""
Stage 1 (批量)：YOLO tracking → 保存轨迹坐标到 CSV
"""
# ── 参数设置 ──────────────────────────────────────────────────────
FOLDER       = "data/video/sample_video"
TRACKS_DIR   = "data/video/sample_video_tracks"
model_name   = "yolo26n.pt"
tracker      = "botsort.yaml"
FRAME_STRIDE = 8  # also works: 1, 2, 4, 8
IMGSZ        = 1920
CONF         = 0.1
IOU          = 0.7
CLASSES      = [0]
MAX_DET      = 300
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
HALF         = DEVICE == "cuda"
# ─────────────────────────────────────────────────────────────────


def print_progress(frame_idx, total, t_start, written):
    elapsed  = time.time() - t_start
    progress = frame_idx / total
    eta      = elapsed / progress * (1 - progress) if progress > 0 else 0
    print(f"  {frame_idx}/{total} ({progress*100:.1f}%)  "
          f"elapsed {elapsed/60:.1f}min  ETA {eta/60:.1f}min  "
          f"rows {written}")


def process_video(model, video_path, output_dir, tracker="botsort.yaml",
                  imgsz=640, device=None, half=False, frame_stride=1,
                  conf=0.25, iou=0.7, classes=None, max_det=300):
    """处理单个视频，用YOLO tracking保存轨迹到CSV。返回写入行数，跳过返回-1。"""
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    model_tag  = os.path.splitext(os.path.basename(model.model_name if hasattr(model, "model_name") else model_name))[0]
    csv_path   = os.path.join(output_dir,
                     f"{video_name}_{model_tag}_conf{conf}_stride{frame_stride}_tracks.csv")

    if os.path.exists(csv_path):
        print(f"  跳过（已存在）: {video_name}")
        return -1

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  ⚠ 无法打开，跳过: {video_path}")
        return -1

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"  {video_name}  {w}×{h} {fps:.1f}fps {total}帧")

    frame_idx = 0
    written   = 0
    t_start   = time.time()

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["frame", "track_id", "cx", "cy", "x1", "y1", "x2", "y2"])

        while cap.isOpened():
            success = cap.grab()
            if not success:
                break

            frame_idx += 1
            if frame_idx % frame_stride != 0:
                continue

            ret, frame = cap.retrieve()
            if not ret:
                break

            result = model.track(frame, imgsz=imgsz, conf=conf, iou=iou,
                                 classes=classes, max_det=max_det,
                                 tracker=tracker, persist=True,
                                 device=device, half=half,
                                 verbose=False)[0]

            if result.boxes is not None and result.boxes.id is not None:
                boxes     = result.boxes.xyxy.cpu().numpy()
                track_ids = result.boxes.id.int().cpu().tolist()
                for box, tid in zip(boxes, track_ids):
                    x1, y1, x2, y2 = box
                    writer.writerow([frame_idx, tid,
                                     round((x1+x2)/2, 1), round(y2, 1),
                                     round(x1,1), round(y1,1), round(x2,1), round(y2,1)])
                    written += 1
                cv2.imshow("preview", result.plot())
                cv2.waitKey(1)

            if frame_idx % (frame_stride * 100) == 0:
                print_progress(frame_idx, total, t_start, written)

    cv2.destroyAllWindows()
    cap.release()
    elapsed = time.time() - t_start
    print(f"  完成: {written} 条记录 → {csv_path}  ({elapsed/60:.1f}min)\n")
    return written


# ── 主流程 ────────────────────────────────────────────────────────
os.makedirs(TRACKS_DIR, exist_ok=True)

VIDEO_LIST = [
    os.path.join(FOLDER, f)
    for f in sorted(os.listdir(FOLDER))
    if f.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))
]

model = YOLO(model_name)
print(f"Device: {DEVICE}  |  共 {len(VIDEO_LIST)} 个视频\n")

for idx, video_path in enumerate(VIDEO_LIST, 1):
    print(f"[{idx}/{len(VIDEO_LIST)}]", end=" ")
    process_video(model, video_path, TRACKS_DIR,
                  tracker=tracker, imgsz=IMGSZ, device=DEVICE, half=HALF,
                  frame_stride=FRAME_STRIDE, conf=CONF, iou=IOU,
                  classes=CLASSES, max_det=MAX_DET)
    model.predictor = None  # 重置tracker状态，避免ID跨视频串联

print("所有视频处理完毕。")

In [ ]:
# https://docs.ultralytics.com/guides/heatmaps/#heatmap-arguments

In [ ]:
"""
Stage 2a (批量)：读取 CSV → 渲染累积轨迹线图
"""
# ── 参数设置 ──────────────────────────────────────────────────────
FOLDER         = "data/video/sample_video"
TRACKS_DIR     = "data/video/sample_video_tracks"
OUTPUT_DIR     = "data/video/sample_video_output"
LINE_THICKNESS = 6
BG_WHITE_MIX   = 0.6
HEAT_OPACITY   = 0.85
# ─────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

_cmap = mcolors.LinearSegmentedColormap.from_list(
    'white_orange_red',
    [(1,1,1), (1,0.95,0.6), (1,0.6,0.0), (0.65,0.1,0.0)], N=256
)
_lut = (_cmap(np.linspace(0, 1, 256))[:, :3] * 255).astype(np.uint8)[:, ::-1]

VIDEO_LIST = [
    os.path.join(FOLDER, f)
    for f in sorted(os.listdir(FOLDER))
    if f.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))
]
print(f"共 {len(VIDEO_LIST)} 个视频\n")

for idx, video_path in enumerate(VIDEO_LIST, 1):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    csv_path   = os.path.join(TRACKS_DIR, f"{video_name}_tracks.csv")
    save_path  = os.path.join(OUTPUT_DIR,  f"{video_name}_tracking_lines_final.jpg")

    if not os.path.exists(csv_path):
        print(f"[{idx}/{len(VIDEO_LIST)}] ⚠ 找不到 CSV，跳过: {video_name}")
        continue
    if os.path.exists(save_path):
        print(f"[{idx}/{len(VIDEO_LIST)}] 跳过（已存在）: {video_name}")
        continue

    print(f"[{idx}/{len(VIDEO_LIST)}] 渲染中: {video_name}")
    df = pd.read_csv(csv_path)
    print(f"  {len(df)} detections, {df['track_id'].nunique()} tracks")

    cap = cv2.VideoCapture(video_path)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ret, background_frame = cap.read()
    cap.release()

    if not ret:
        print(f"  ⚠ 无法读取背景帧，跳过")
        continue

    acc_map   = np.zeros((h, w), dtype=np.float32)
    temp_line = np.zeros((h, w), dtype=np.uint8)

    for tid, group in df.groupby("track_id"):
        pts = group.sort_values("frame")[["cx", "cy"]].values
        for i in range(len(pts) - 1):
            pt1 = (int(np.clip(pts[i,   0], 0, w-1)), int(np.clip(pts[i,   1], 0, h-1)))
            pt2 = (int(np.clip(pts[i+1, 0], 0, w-1)), int(np.clip(pts[i+1, 1], 0, h-1)))
            temp_line[:] = 0
            cv2.line(temp_line, pt1, pt2, 1, thickness=LINE_THICKNESS)
            acc_map += temp_line

    log_map  = np.log1p(acc_map)
    norm_map = (log_map / log_map.max() * 255).astype(np.uint8) if log_map.max() > 0 else np.zeros((h, w), dtype=np.uint8)
    white    = np.full((h, w, 3), 255, dtype=np.uint8)

    bg         = cv2.addWeighted(background_frame, 1 - BG_WHITE_MIX, white, BG_WHITE_MIX, 0)
    heat_color = _lut[norm_map]
    alpha      = (norm_map.astype(np.float32) / 255.0 * HEAT_OPACITY)[:, :, np.newaxis]
    output     = (alpha * heat_color + (1 - alpha) * bg).astype(np.uint8)

    cv2.imwrite(save_path, output)
    print(f"  保存: {save_path}")
    ipython_display(PILImage.fromarray(cv2.cvtColor(output, cv2.COLOR_BGR2RGB)))

print("\n所有视频渲染完毕。")

In [ ]:
"""
Stage 2b (批量)：读取 CSV → 渲染动态热力图视频 + 静态热力图
"""
# ── 参数设置 ──────────────────────────────────────────────────────
FOLDER        = "data/video/sample_video"
TRACKS_DIR    = "data/video/sample_video_tracks"
OUTPUT_DIR    = "data/video/sample_video_output"

MODEL_TAG     = "yolo26n"  # 与 Stage 1 一致
CONF          = 0.1
FRAME_STRIDE  = 1

COLORMAP      = cv2.COLORMAP_JET
BG_WEIGHT     = 0.5
HEAT_WEIGHT   = 0.5
CIRCLE_RADIUS = None  # None=用检测框内切圆，或指定固定像素半径如15
# ─────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

VIDEO_LIST = [
    os.path.join(FOLDER, f)
    for f in sorted(os.listdir(FOLDER))
    if f.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))
]


def heatmap_effect(heatmap, x0, y0, x1, y1):
    x0, y0 = max(0, x0), max(0, y0)
    x1, y1 = min(heatmap.shape[1], x1), min(heatmap.shape[0], y1)
    if x1 <= x0 or y1 <= y0:
        return
    if CIRCLE_RADIUS is not None:
        cx, cy = (x0+x1)//2, (y0+y1)//2
        cv2.circle(heatmap, (cx, cy), CIRCLE_RADIUS, 2.0, -1)
    else:
        radius_sq = (min(x1-x0, y1-y0) // 2) ** 2
        xv, yv = np.meshgrid(np.arange(x0, x1), np.arange(y0, y1))
        cx, cy = (x0+x1)//2, (y0+y1)//2
        within = (xv-cx)**2 + (yv-cy)**2 <= radius_sq
        heatmap[y0:y1, x0:x1][within] += 2


def render_heatmap(heatmap, bg):
    norm    = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    colored = cv2.applyColorMap(norm, COLORMAP)
    return cv2.addWeighted(bg, BG_WEIGHT, colored, HEAT_WEIGHT, 0)


for idx, video_path in enumerate(VIDEO_LIST, 1):
    video_name     = os.path.splitext(os.path.basename(video_path))[0]
    csv_path       = os.path.join(TRACKS_DIR, f"{video_name}_{MODEL_TAG}_conf{CONF}_stride{FRAME_STRIDE}_tracks.csv")
    save_path      = os.path.join(OUTPUT_DIR,  f"{video_name}_heatmap.jpg")
    video_path_out = os.path.join(OUTPUT_DIR,  f"{video_name}_heatmap.mp4")

    if not os.path.exists(csv_path):
        print(f"[{idx}/{len(VIDEO_LIST)}] ⚠ 找不到CSV，跳过: {video_name}")
        continue

    print(f"[{idx}/{len(VIDEO_LIST)}] 渲染中: {video_name}")
    df = pd.read_csv(csv_path)
    print(f"  {len(df)} detections, {df['track_id'].nunique()} tracks")

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    _, bg0 = cap.read()
    h, w = bg0.shape[:2]

    heatmap = np.zeros((h, w), dtype=np.float32)
    writer  = cv2.VideoWriter(video_path_out,
                              cv2.VideoWriter_fourcc(*"mp4v"),
                              fps / FRAME_STRIDE, (w, h))

    for frame_idx, frame_df in df.groupby("frame"):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx - 1)
        ret, bg = cap.read()
        if not ret:
            bg = bg0
        for _, row in frame_df.iterrows():
            heatmap_effect(heatmap,
                           int(row["x1"]), int(row["y1"]),
                           int(row["x2"]), int(row["y2"]))
        writer.write(render_heatmap(heatmap, bg))

    cap.release()
    writer.release()

    output = render_heatmap(heatmap, bg)
    cv2.imwrite(save_path, output)
    print(f"  静态图: {save_path}")
    print(f"  过程视频: {video_path_out}")
    ipython_display(PILImage.fromarray(cv2.cvtColor(output, cv2.COLOR_BGR2RGB)))

print("\n所有视频渲染完毕。")

In [ ]:
"""
Stage 2c (批量)：读取 CSV → 渲染简单热力图（旧版，点状累积）
"""
# ── 参数设置 ──────────────────────────────────────────────────────
FOLDER     = "data/video/sample_video"
TRACKS_DIR = "data/video/sample_video_tracks"
OUTPUT_DIR = "data/video/sample_video_output"
# ─────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

VIDEO_LIST = [
    os.path.join(FOLDER, f)
    for f in sorted(os.listdir(FOLDER))
    if f.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))
]

for idx, video_path in enumerate(VIDEO_LIST, 1):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    csv_path   = os.path.join(TRACKS_DIR, f"{video_name}_tracks.csv")
    save_path  = os.path.join(OUTPUT_DIR,  f"{video_name}_heatmap.jpg")

    if not os.path.exists(csv_path):
        print(f"[{idx}/{len(VIDEO_LIST)}] ⚠ 找不到CSV，跳过: {video_name}")
        continue
    if os.path.exists(save_path):
        print(f"[{idx}/{len(VIDEO_LIST)}] 跳过（已存在）: {video_name}")
        continue

    print(f"[{idx}/{len(VIDEO_LIST)}] 渲染中: {video_name}")
    df = pd.read_csv(csv_path)
    print(f"  {len(df)} detections, {df['track_id'].nunique()} tracks")

    cap = cv2.VideoCapture(video_path)
    _, bg = cap.read()
    cap.release()
    h, w = bg.shape[:2]

    heatmap = np.zeros((h, w), dtype=np.float32)
    for _, row in df.iterrows():
        cx = int(np.clip(row["cx"], 0, w-1))
        cy = int(np.clip(row["cy"], 0, h-1))
        cv2.circle(heatmap, (cx, cy), 15, 1.0, -1)

    norm    = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    colored = cv2.applyColorMap(norm, cv2.COLORMAP_JET)
    output  = cv2.addWeighted(bg, 0.5, colored, 0.5, 0)

    cv2.imwrite(save_path, output)
    print(f"  保存: {save_path}")
    ipython_display(PILImage.fromarray(cv2.cvtColor(output, cv2.COLOR_BGR2RGB)))

print("\n所有视频渲染完毕。")

In [1]:
"""
工具：批量裁剪视频前 N 分钟
"""
import subprocess

INPUT_DIR  = "/Volumes/SSD/huangqiangbei-walking/video-footage"
OUTPUT_DIR = "/Volumes/SSD/huangqiangbei-walking/video-footage-clipped"
CLIP_MIN   = 10  # 保留前几分钟

os.makedirs(OUTPUT_DIR, exist_ok=True)

VIDEO_LIST = [
    os.path.join(INPUT_DIR, f)
    for f in sorted(os.listdir(INPUT_DIR))
    if f.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))
]
print(f"共 {len(VIDEO_LIST)} 个视频，裁剪前 {CLIP_MIN} 分钟\n")

for idx, src in enumerate(VIDEO_LIST, 1):
    fname = os.path.basename(src)
    dst   = os.path.join(OUTPUT_DIR, fname)

    if os.path.exists(dst):
        print(f"[{idx}/{len(VIDEO_LIST)}] 跳过（已存在）: {fname}")
        continue

    print(f"[{idx}/{len(VIDEO_LIST)}] 处理中: {fname}")
    cmd = [
        "ffmpeg", "-y",
        "-i", src,
        "-t", str(CLIP_MIN * 60),
        "-c", "copy",          # 直接复制流，不重新编码，极快
        dst
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  → {dst}")
    else:
        print(f"  ⚠ 失败: {result.stderr.splitlines()[-1]}")

print("\n全部完成。")

NameError: name 'os' is not defined